# Explicabilidad y simulador de riesgo por vano

Hermano de `04_uiti_vano_trayectorias_vano.ipynb`: reutiliza su misma clasificacion
KMeans por vano x ventana (nunca se reajusta aca) y agrega, sobre esa base, un
simulador interactivo de "que pasaria si" a nivel de vano usando el modelo MGCECDL.

**Requiere un kernel Python vivo.** Este cuaderno NO es exportable a HTML estatico ni
publicable como Databricks App, a diferencia de 02/03/04: sus controles
(`ipywidgets`) y el simulador corren en el kernel, no en el navegador.

**Que mide cada mapa (no confundir).** **Criticidad Original** (fila 1) es el grupo
historico: la clase KMeans que 04 ya calculo sobre eventos observados. **Criticidad
Simulada** (fila 2) es la clase que el modelo MGCECDL predice al aplicar las variables
del simulador. Son dos mediciones distintas y nunca comparten leyenda ni titulo.

**Un solo disparador.** El boton "Simular" hace las dos cosas de una vez: pinta el mapa
simulado y calcula **Importancia Variables**, el barrido de sensibilidad min-max
normalizado con softmax sobre las muestras de los vanos marcados del circuito y la
ventana activos (sin vanos marcados, sobre el circuito completo en esa ventana).

**Estado actual**: la figura de 3x2 paneles con sus 28 trazas funciona de punta a punta:
los dos mapas a ancho completo, y en la fila 3 la importancia de variables y la nube
KMeans de grupos de vanos. El panel del grafo reconstruido (decision D4) sigue diferido
-- su traza existe, esta indexada, y se muestra oculta.

In [ ]:
import asyncio
import sys
import time
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

try:
    import ipywidgets as widgets
except ImportError as exc:
    raise ImportError('Este cuaderno requiere ipywidgets para la interfaz interactiva.') from exc
from IPython.display import display

# Sube desde el cwd hasta la raiz del repo (marcada por la carpeta src/), igual que 09.
# Se agregan ROOT y ROOT/src -- no solo src/ -- porque ventanas_015.py importa
# `scripts.extract_geometrias_014` (paquete de nivel de repo, igual que en notebook 10).
ROOT = Path.cwd().resolve()
while not (ROOT / 'src').is_dir() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
for _path_a_agregar in (ROOT, ROOT / 'src'):
    if str(_path_a_agregar) not in sys.path:
        sys.path.insert(0, str(_path_a_agregar))

# Un kernel que ya importo estos paquetes se queda con la version VIEJA en `sys.modules`:
# "Run All" sin reiniciar NO vuelve a leer el disco. Un rename en src/ -- por ejemplo
# `SelectorVanos._caja` -> `.caja` -- estalla entonces como AttributeError diez celdas mas
# abajo, con el codigo del disco ya correcto. Se purgan ANTES de importarlos, asi el
# cuaderno corre SIEMPRE contra la fuente actual, con o sin reinicio de kernel. Va aca y no
# como `importlib.reload`: reload no rehace los objetos ya construidos con la clase vieja,
# y este cuaderno los reconstruye todos de esta celda para abajo.
for _modulo in [m for m in list(sys.modules)
                if m.split('.')[0] in ('chec_impacto', 'chec_local_interpreter', 'scripts')]:
    del sys.modules[_modulo]

from chec_impacto.data import preparar_splits_estratificados, procesar_dataset_completo
from chec_impacto.models.criticality_assignment import (
    CLAVE_ESPACIO_CANONICO,
    GEOMETRIAS_SHA1_ESPERADO,
    cargar_geometria_014,
    verificar_sha1_geometrias,
)
from chec_impacto.training import (
    cargar_modelo_mgcecdl,
    escalar_features_minmax_mgcecdl,
    predict_classification,
    resolve_training_device,
)
from chec_local_interpreter.vano_controls import build_knobs, expand_knob_overrides
from chec_local_interpreter.vano_widgets import (
    construir_selector_casillas,
    construir_selector_vanos,
)
from chec_local_interpreter.ventanas_015 import (
    capas_mapa_historico,
    cargar_clases_desde_014,
    centro_y_zoom,
    construir_hist_class_cache,
    construir_mask_cache,
    construir_tabla_vano_ventana,
    construir_ventanas,
    fid_de_punto,
    nube_fondo,
    nube_seleccion,
)
from scripts.extract_geometrias_014 import (
    DEFAULT_NOTEBOOK_PATH,
    DEFAULT_OUTPUT_PATH,
    extraer_geometrias_014,
)

# Sonda del contrato que rompio el cuaderno dos veces. Si el kernel siguiera sirviendo una
# version vieja de vano_widgets, falla ACA -- primera celda, mensaje que dice que hacer --
# en vez de a los 10 minutos de procesamiento, en la celda del panel.
_sonda = construir_selector_vanos(['0'])
assert hasattr(_sonda, 'caja'), (
    'vano_widgets viejo en memoria: el selector de casillas sin `.caja`. '
    'Reinicia el kernel. '
    f'(modulo cargado desde {sys.modules["chec_local_interpreter.vano_widgets"].__file__})'
)
del _sonda

In [ ]:
# Ventana climatica igual que 03_mgcecdl_training / 09_simulador: cambiarla generaria un
# set de features distinto al que el modelo cargado en la celda SEAM espera.
VENTANA_CLIMATICA_HORAS = 12
CLAVE_ESPACIO = CLAVE_ESPACIO_CANONICO  # '2' -- espacio canonico fijado en criticality_assignment.py
DEVICE = resolve_training_device('auto')

# Misma paleta que 01.4: los grupos historicos de este cuaderno SON los de 01.4, nunca se
# reajustan, asi que el color tiene que significar lo mismo en los dos cuadernos.
NOMBRES_GRUPOS = ['Bajo', 'Medio', 'Medio-Alto', 'Alto']
COLORES_GRUPOS = ['rgb(252,187,161)', 'rgb(251,106,74)', 'rgb(203,24,29)', 'rgb(103,0,13)']
COLOR_SIN_DATO = '#94a3b8'  # mismo gris "sin grupo" de 01.3/01.4, distinto de los 4 colores
COLOR_AUN_NO_SIMULADO = '#a78bfa'  # violeta, distinto de COLOR_SIN_DATO: fila 2 antes de
# la primera simulacion no comparte color/leyenda con "simulado, sin filas de evento" (D2, hallazgo W1)
COLOR_MARCADO = '#0072b2'
# Equipos: mismos colores que 01.4, por el mismo motivo que la paleta de grupos --
# un naranja tiene que seguir siendo un transformador al pasar de un cuaderno a otro.
COLOR_TRAFO = '#f59e0b'
COLOR_SWITCH = '#7c3aed'
ANCHO_MAPA = 3.0
ANCHO_MAPA_MARCADO = round(ANCHO_MAPA * 1.4, 2)

# Fila 1, paridad 01.4: un vano MARCADO se dibuja con el color de SU clase, sobre un halo
# blanco que lo despega del fondo (01.4: `width=ANCHO_MAPA_RESALTE * 2.6, color='white'`).
# Un color plano de "seleccionado" encima de la clase congela lo que se ve: la ventana
# cambia la clase por debajo y el vano marcado sigue igual en pantalla. COLOR_MARCADO
# queda solo para la fila 2, donde la clase la pone el modelo y no el KMeans.
COLOR_HALO = 'white'
ANCHO_HALO = round(ANCHO_MAPA_MARCADO * 2.6, 2)
# Negro, el mismo `COLOR_SIN_EVENTO = 'rgb(0,0,0)'` de 01.4: el vano marcado que no tiene
# celda en la ventana (sin eventos, o ausente de la tabla) no es la clase mas baja.
COLOR_MARCADO_SIN_DATO = 'rgb(0,0,0)'
OPACIDAD_NUBE = 0.45               # 01.4, para que la nube de fondo no tape el resaltado


In [ ]:
# --- Reutilizacion de la geometria KMeans de 01.4 (design section F) -------
# Falla RAPIDO aca, antes de procesar el dataset completo (celda siguiente): si 01.4 fue
# editado y sus centroides se movieron, no tiene sentido esperar el procesamiento pesado
# para enterarse. `cargar_clases_desde_014` (celda 7, via hist_class_cache) repite esta
# misma verificacion por cada ventana consultada -- barata, y evita que una geometria
# cacheada quede sin recomprobar dentro de la misma sesion.
GEOMETRIAS_PATH = DEFAULT_OUTPUT_PATH
if not GEOMETRIAS_PATH.exists():
    extraer_geometrias_014(DEFAULT_NOTEBOOK_PATH, GEOMETRIAS_PATH)
_sha1_real, _coincide = verificar_sha1_geometrias(GEOMETRIAS_PATH, esperado=GEOMETRIAS_SHA1_ESPERADO)
assert _coincide, (
    f'La geometria KMeans extraida de 01.4 no coincide con la esperada '
    f'(esperado={GEOMETRIAS_SHA1_ESPERADO}, real={_sha1_real}). 01.4 fue modificado; '
    f'01.5 depende de esa geometria.'
)
# La geometria en si (no solo su sha1): la nube KMeans de la fila 3 tiene que dibujarse en
# el MISMO espacio en que se asignan las clases -- el canonico '2' es (log_x=False,
# log_y=True). Leerlo de la geometria y no fijarlo a mano evita que un cambio de espacio
# deje la nube en ejes que ya no corresponden a las fronteras.
GEOMETRIA_014 = cargar_geometria_014(GEOMETRIAS_PATH, CLAVE_ESPACIO)
print(f'Geometria 01.4 verificada -- sha1 coincide ({_sha1_real[:12]}...) | '
      f'espacio {CLAVE_ESPACIO}: log_x={GEOMETRIA_014.logs[0]}, log_y={GEOMETRIA_014.logs[1]}')

In [ ]:
DATA_PATH = ROOT / 'data' / 'Indicadores_vano_v3.csv'
VARIABLES_SELECCION_PATH = ROOT / 'data' / 'Variables_seleccion.xlsx'
MODEL_DIR = ROOT / 'data' / 'models'

# Mismo preprocesamiento real usado en entrenamiento (03_mgcecdl_training / 09_simulador):
# sin muestreo ni filtro de UITI, para que context_df quede alineado FILA A FILA con X --
# la clave que permite reusar la MISMA mascara (circuito, ventana) para el mapa historico
# (sin modelo) y, en un PR futuro, para las predicciones del modelo sobre esas mismas filas.
datos = procesar_dataset_completo(
    path_clima=DATA_PATH,
    path_variables_seleccion=VARIABLES_SELECCION_PATH,
    use_sampling=False,
    min_samples_per_codigo=5,
    target='UITI_VANO',
    filtro_uiti_max=None,
    ventana_climatica_horas=VENTANA_CLIMATICA_HORAS,
)

feature_names = list(datos['features'])
X_raw_model = np.asarray(datos['X'], dtype=np.float32)
Xdf = datos['Xdata'].copy().reset_index(drop=True)
context_df = datos['df_original_copy'].copy().reset_index(drop=True)
label_encoders = datos.get('label_encoders', {})
max_values_imputed = datos.get('max_values_imputed', {})

splits_clf = escalar_features_minmax_mgcecdl(
    preparar_splits_estratificados(
        X_raw_model, datos['y'], modo='clasificacion', random_state=42,
    )
)
feature_scaler = splits_clf['feature_scaler']
X = feature_scaler.transform(X_raw_model).astype(np.float32)

assert len(context_df) == len(X), 'context_df y X deben quedar alineados fila a fila'
print(f'{len(context_df):,} filas | {len(feature_names)} features | X{X.shape}')

In [ ]:
# --- SEAM D1: cambiar SOLO este bloque para pasar a MIL (design section D) -------------
MODEL_PATH = MODEL_DIR / 'mgcecdl_classifier_best.zip'  # nombre FIJO, no "el mas reciente":
# evita levantar por error un checkpoint todavia en curso de otra sesion en este repo.
MODELO = cargar_modelo_mgcecdl(str(MODEL_PATH), device=DEVICE)
PREDICT_FN = predict_classification
# MIL (PR futuro):
# MODELO = BagPredictor(mil_model, feature_names=FEATURES_MIL, geometria=GEOMETRIA)
# PREDICT_FN = mil_vano_ventana.predict_fn

_probe = PREDICT_FN(MODELO, X[: min(len(X), 8)], device=DEVICE, batch_size=8)
assert set(_probe) >= {'fused_probs', 'predicted_classes'}
N_CLASSES = int(np.asarray(_probe['fused_probs']).shape[1])
print(f'Modelo cargado -- {N_CLASSES} clases (contrato PREDICT_FN verificado)')

In [ ]:
# --- construir_ventanas + per-(vano, ventana) events + caches (design section A) -------
VENTANAS = construir_ventanas(context_df['FECHA'])
TABLA = construir_tabla_vano_ventana(context_df, VENTANAS)
mask_para = construir_mask_cache(TABLA)
clases_para = construir_hist_class_cache(TABLA, mask_para)

# La clase de CADA celda (vano x ventana) de una sola pasada: es la misma asignacion por
# centroide mas cercano que hace `clases_para` ventana por ventana (es puntual, fila a
# fila), pero calculada una vez para poder dibujar la nube KMeans completa de la fila 3.
CLASE_TABLA, _n_clamped = cargar_clases_desde_014(
    TABLA['num_eventos'].to_numpy(dtype=float),
    TABLA['uiti_acumulado'].to_numpy(dtype=float),
)
NUBE_FONDO = nube_fondo(TABLA, CLASE_TABLA)
print(f'nube KMeans: {len(TABLA):,} celdas | por clase '
      f'{[len(c["x"]) for c in NUBE_FONDO]} | {_n_clamped} valores recortados por eps')

CIRCUITOS = sorted(TABLA['CIRCUITO'].astype(str).unique())
VANOS_POR_CIRCUITO = {
    c: sorted(g['FID_VANO'].unique().tolist())
    for c, g in TABLA.groupby(TABLA['CIRCUITO'].astype(str))
}

print(f'{len(TABLA):,} celdas vano x ventana con eventos | {len(VENTANAS)} ventanas | '
      f'{TABLA["FID_VANO"].nunique():,} vanos distintos | {len(CIRCUITOS)} circuitos')


# Geometria FISICA de cada vano (no confundir con la geometria KMeans de la celda 4): mismo
# shapefile y mismo join que el mapa de 01.3/01.4. No se extrae a src/ porque es solo
# lectura + reindexado geoespacial, sin logica propia que valga la pena testear por fuera
# de lo que TABLA/capas_mapa_historico ya cubren.
def _norm_id(serie):
    return (serie.astype('string').str.strip().str.replace(r'\.0$', '', regex=True)
            .replace({'': pd.NA, '<NA>': pd.NA, 'nan': pd.NA, 'None': pd.NA}))


_lineas = gpd.read_file(ROOT / 'data' / 'GEO' / 'MVLINSEC.shp')
if str(_lineas.crs) != 'EPSG:4326':
    _lineas = _lineas.to_crs('EPSG:4326')
_lineas['FID_VANO_GEO'] = _norm_id(_lineas['G3E_FID'])
_utiles = _lineas[_lineas['CIRCUITO'].astype(str).isin(set(CIRCUITOS))]

GEO_POR_CIRCUITO = {}
for _c, _g in _utiles.groupby(_utiles['CIRCUITO'].astype(str)):
    fids, lats, lons = [], [], []
    for _fid, _geom in zip(_g['FID_VANO_GEO'], _g.geometry):
        if _geom is None or _geom.is_empty:
            continue
        for _p in ([_geom] if _geom.geom_type == 'LineString' else list(getattr(_geom, 'geoms', []))):
            xs, ys = _p.xy
            fids.append(str(_fid))
            lats.append([round(v, 5) for v in ys])
            lons.append([round(v, 5) for v in xs])
    if fids:
        # `bounds` es lo que permite encuadrar el mapa sobre el circuito elegido, igual
        # que 01.4: [lat_min, lat_max, lon_min, lon_max].
        _la = [v for l in lats for v in l]
        _lo = [v for l in lons for v in l]
        GEO_POR_CIRCUITO[_c] = {
            'fids': fids, 'lat': lats, 'lon': lons,
            'bounds': [round(min(_la), 5), round(max(_la), 5),
                       round(min(_lo), 5), round(max(_lo), 5)],
        }


def _equipo(nombre):
    """Transformadores e interruptores del circuito, igual que 01.4 celda 5. Si el
    shapefile no esta, el mapa se dibuja sin equipos en vez de fallar: son contexto
    de lectura, no el dato del tablero."""
    ruta = ROOT / 'data' / 'GEO' / nombre
    if not ruta.exists():
        return {}
    g = gpd.read_file(ruta)
    if str(g.crs) != 'EPSG:4326':
        g = g.to_crs('EPSG:4326')
    g = g[g['CIRCUITO'].astype(str).isin(set(CIRCUITOS))]
    g = g[g.geometry.notna() & ~g.geometry.is_empty]
    return {c: {'lat': [round(float(p.y), 5) for p in gg.geometry],
                'lon': [round(float(p.x), 5) for p in gg.geometry]}
            for c, gg in g.groupby(g['CIRCUITO'].astype(str))}


TRAFOS = _equipo('GDBCHEC_TRANSFOR.shp')
SWITCHES = _equipo('SWITCHES.shp')

# UITI y eventos por vano y ventana: solo alimentan el hover, igual que 01.4. El grupo
# NO se guarda aca -- sale de `clases_para`, que es la unica fuente de clases.
DATOS_VENTANA = [{} for _ in VENTANAS]
for _fid, _vi, _u, _n in zip(TABLA['FID_VANO'], TABLA['ventana_i'],
                             TABLA['uiti_acumulado'], TABLA['num_eventos']):
    DATOS_VENTANA[int(_vi)][str(_fid)] = (float(_u), int(_n))

print(f'{len(GEO_POR_CIRCUITO)} circuitos con geometria fisica | '
      f'{sum(len(v["lat"]) for v in TRAFOS.values()):,} transformadores | '
      f'{sum(len(v["lat"]) for v in SWITCHES.values()):,} switches')

In [ ]:
KNOBS = build_knobs(
    feature_names=feature_names,
    original_feature_df=Xdf,
    label_encoders=label_encoders,
    max_values_imputed=max_values_imputed,
)
print(f'{len(KNOBS)} controles (Knob catalog, PR2a) -- '
      f'{sum(1 for k in KNOBS if k.kind == "categorical")} categoricos, '
      f'{sum(1 for k in KNOBS if k.kind == "numeric")} numericos, '
      f'{sum(1 for k in KNOBS if k.kind == "constant")} constantes')

In [ ]:
# --- Inventario de trazas CONGELADO (design section G). Los indices 0-17 no se mueven:
# la grilla cambio de 2x3 a 3x2 y ninguna traza cambio de posicion en la LISTA, solo de
# subplot. El grafo reconstruido (decision D4) sigue mutando el indice 7 en un PR futuro,
# con sus trazas nuevas a partir del 18.
# Los mapas ocupan las DOS columnas de su fila (colspan=2): un mapa geografico compartido
# entre 4 filas de vanos y una barra de importancia no se lee, y estirarlo al ancho
# completo es lo unico que permite distinguir tramos vecinos. Las otras dos figuras --
# importancia de variables y el panel reservado -- bajan juntas a la fila 3.
# Los equipos (14-17) van ultimos por orden de dibujo: los marcadores tienen que quedar
# por encima de las lineas, como en 01.4.
IDX = {
    'clases': [0, 1, 2, 3],          # fila 1 (2 columnas) -- mapa historico (01.4), PR3
    'sin_dato': 4,                    # fila 1 -- sin eventos en la ventana
    'marcados': 5,                    # fila 1 -- halo de vanos marcados
    'ranking': 6,                     # fila 3 col 1 -- importancia de variables, PR4
    'diferido': 7,                    # RESERVADO (oculta) -- decision D4
    'pred_clases': [8, 9, 10, 11],    # fila 2 (2 columnas) -- mapa predicho MGCECDL, PR5
    'pred_sin_dato': 12,               # fila 2, PR5
    'pred_marcados': 13,               # fila 2, PR5
    'trafos': 14,                      # fila 1 -- equipos, PR6
    'switches': 15,                    # fila 1 -- equipos, PR6
    'pred_trafos': 16,                 # fila 2 -- equipos, PR6
    'pred_switches': 17,               # fila 2 -- equipos, PR6
    # Nuevas, a partir del 18 y sin mover ninguna anterior (design section G).
    'marcados_clases': [18, 19, 20, 21],  # fila 1 -- marcado, con el color de SU clase
    'marcados_sin_dato': 22,              # fila 1 -- marcado sin celda en la ventana: negro
    'nube_clases': [23, 24, 25, 26],      # fila 3 col 2 -- nube KMeans (fondo fijo)
    'nube_seleccion': 27,                 # fila 3 col 2 -- celdas de lo marcado
}

_fig = make_subplots(
    rows=3, cols=2,
    specs=[[{'type': 'map', 'colspan': 2}, None],
           [{'type': 'map', 'colspan': 2}, None],
           [{'type': 'xy'}, {'type': 'xy'}]],
    # Un titulo por subplot REAL: las celdas `None` del colspan no consumen ninguno.
    subplot_titles=(
        'Criticidad Original',
        'Criticidad Simulada',
        'Importancia Variables',
        'Grupos KMeans de vanos',
    ),
    row_heights=[0.37, 0.37, 0.26],
    horizontal_spacing=0.10, vertical_spacing=0.07,
)

for _clase in range(4):                                          # 0-3
    _fig.add_trace(go.Scattermap(
        lat=[], lon=[], mode='lines', name=NOMBRES_GRUPOS[_clase],
        legendgroup='hist', legendgrouptitle_text='Criticidad original',
        line=dict(width=ANCHO_MAPA, color=COLORES_GRUPOS[_clase]),
        hovertext=[], hoverinfo='text',
    ), row=1, col=1)
_fig.add_trace(go.Scattermap(                                    # 4
    lat=[], lon=[], mode='lines', name='Sin dato', legendgroup='hist',
    line=dict(width=ANCHO_MAPA, color=COLOR_SIN_DATO),
    hovertext=[], hoverinfo='text',
), row=1, col=1)
# El marcado de la fila 1 va en DOS capas, como en 01.4: primero el halo blanco ancho
# (esta traza, la 5), y despues -- indices 18-22, al final por orden de dibujo -- la linea
# con el color de la clase del vano. El halo no va a la leyenda: una linea blanca sobre
# fondo blanco no dice nada ahi.
_fig.add_trace(go.Scattermap(                                    # 5
    lat=[], lon=[], mode='lines', name='Vano marcado', legendgroup='hist',
    showlegend=False,
    line=dict(width=ANCHO_HALO, color=COLOR_HALO),
    hovertext=[], hoverinfo='text',
), row=1, col=1)

_fig.add_trace(go.Bar(x=[], y=[], orientation='h', showlegend=False,
                      hovertext=[], hoverinfo='text'), row=3, col=1)  # 6
# La barra ya no muestra la magnitud cruda sino su participacion softmax: un porcentaje
# se lee sin conocer las unidades del modelo, una "sensibilidad min-max" de 0.0143 no.
_fig.update_xaxes(title_text='Relevancia (softmax)', tickformat='.0%', row=3, col=1)
_fig.update_yaxes(tickfont=dict(size=10), row=3, col=1)

# La 7 sigue existiendo y sigue oculta: su indice esta congelado y el panel del grafo
# reconstruido (decision D4) la va a usar. Ahora comparte subplot con la nube.
_fig.add_trace(go.Scatter(x=[], y=[], mode='markers', visible=False, showlegend=False), row=3, col=2)  # 7

for _clase in range(4):                                          # 8-11
    _fig.add_trace(go.Scattermap(
        lat=[], lon=[], mode='lines', name=NOMBRES_GRUPOS[_clase],
        legendgroup='pred', legendgrouptitle_text='Criticidad simulada', showlegend=False,
        line=dict(width=ANCHO_MAPA, color=COLORES_GRUPOS[_clase]),
        hovertext=[], hoverinfo='text',
    ), row=2, col=1)
_fig.add_trace(go.Scattermap(                                    # 12
    lat=[], lon=[], mode='lines', name='Sin dato', legendgroup='pred', showlegend=False,
    line=dict(width=ANCHO_MAPA, color=COLOR_SIN_DATO),
    hovertext=[], hoverinfo='text',
), row=2, col=1)
_fig.add_trace(go.Scattermap(                                    # 13
    lat=[], lon=[], mode='lines', name='Vano marcado', legendgroup='pred', showlegend=False,
    line=dict(width=ANCHO_MAPA_MARCADO, color=COLOR_MARCADO),
    hovertext=[], hoverinfo='text',
), row=2, col=1)

# Los equipos van al final para dibujarse ENCIMA de los tramos. Se repiten por fila
# porque una traza pertenece a un solo subplot: no hay forma de compartirla entre los
# dos mapas, y sin ellos la fila 2 se leeria como otra geografia.
for _fila, _leyenda in ((1, True), (2, False)):
    for _nombre, _color, _tam in [('Transformadores', COLOR_TRAFO, 6),
                                  ('Switches', COLOR_SWITCH, 5)]:
        _fig.add_trace(go.Scattermap(                             # 14-17
            lat=[], lon=[], mode='markers', name=_nombre,
            legendgroup='equipos', legendgrouptitle_text='Equipos',
            showlegend=_leyenda,
            marker=dict(size=_tam, color=_color), hovertext=[], hoverinfo='text',
        ), row=_fila, col=1)

# --- 18-22: el marcado de la fila 1, con el color de su clase (paridad 01.4) ---------
for _clase in range(4):                                          # 18-21
    _fig.add_trace(go.Scattermap(
        lat=[], lon=[], mode='lines', name=NOMBRES_GRUPOS[_clase],
        legendgroup='hist', showlegend=False,
        line=dict(width=ANCHO_MAPA_MARCADO, color=COLORES_GRUPOS[_clase]),
        hovertext=[], hoverinfo='text',
    ), row=1, col=1)
_fig.add_trace(go.Scattermap(                                    # 22
    lat=[], lon=[], mode='lines', name='Marcado sin eventos', legendgroup='hist',
    line=dict(width=ANCHO_MAPA_MARCADO, color=COLOR_MARCADO_SIN_DATO),
    hovertext=[], hoverinfo='text',
), row=1, col=1)

# --- 23-27: la nube KMeans de 01.4 en el panel que estaba reservado ------------------
# El fondo son TODAS las celdas (vano x ventana) del dataset, agrupadas por clase: es
# donde estan las fronteras, y no se mueve nunca. Encima, las celdas de lo marcado en la
# ventana activa. Ahi se ve lo que el mapa solo insinua: mover la ventana mueve el punto
# del vano por el plano (eventos, UITI) y por eso cambia su clase.
for _clase in range(4):                                          # 23-26
    _fig.add_trace(go.Scattergl(
        x=[], y=[], mode='markers', name=NOMBRES_GRUPOS[_clase],
        legendgroup='nube', legendgrouptitle_text='Nube KMeans', showlegend=False,
        marker=dict(size=3.5, color=COLORES_GRUPOS[_clase], opacity=OPACIDAD_NUBE),
        hoverinfo='skip',
    ), row=3, col=2)
_fig.add_trace(go.Scattergl(                                     # 27
    x=[], y=[], mode='markers', name='Seleccion', showlegend=False,
    marker=dict(size=9, color=[], line=dict(width=1.4, color='#111111')),
    hovertext=[], hoverinfo='text',
), row=3, col=2)
_fig.update_xaxes(title_text='Eventos en la ventana',
                  type='log' if GEOMETRIA_014.logs[0] else 'linear', row=3, col=2)
_fig.update_yaxes(title_text='UITI acumulado',
                  type='log' if GEOMETRIA_014.logs[1] else 'linear', row=3, col=2)

_fig.update_layout(
    map=dict(style='carto-positron', center=dict(lat=5.07, lon=-75.52), zoom=10),
    map2=dict(style='carto-positron', center=dict(lat=5.07, lon=-75.52), zoom=10),
    title=dict(text='Simulador Criticidad'),
    # Dos mapas apilados a ancho completo piden alto: con los 760 de la grilla 2x3 cada
    # mapa quedaba en una franja de ~300 px y los tramos se pisaban entre si.
    height=1180, width=1280, template='plotly_white',
    legend=dict(y=1.0, yanchor='top'),
)

# Los indices se verifican al generar, igual que 01.3/01.4: si alguien reordena las
# trazas esto falla ACA, no se descubre silenciosamente en la celda de dibujo.
assert len(_fig.data) == 28, len(_fig.data)
assert all(_fig.data[i].type == 'scattermap' for i in IDX['clases'] + [IDX['sin_dato'], IDX['marcados']])
assert [_fig.data[i].line.color for i in IDX['clases']] == COLORES_GRUPOS
assert _fig.data[IDX['ranking']].type == 'bar'
assert _fig.data[IDX['diferido']].type == 'scatter' and _fig.data[IDX['diferido']].visible is False
assert all(_fig.data[i].type == 'scattermap'
           for i in IDX['pred_clases'] + [IDX['pred_sin_dato'], IDX['pred_marcados']])
# Los equipos son PUNTOS y van despues de todas las lineas: si alguien los adelanta,
# quedan tapados por los tramos y esto falla al generar, no en el navegador.
assert all(_fig.data[i].mode == 'markers'
           for i in (IDX['trafos'], IDX['switches'], IDX['pred_trafos'], IDX['pred_switches']))
assert min(IDX['trafos'], IDX['switches'], IDX['pred_trafos'], IDX['pred_switches']) > IDX['pred_marcados']
# El marcado con color de clase va DESPUES del halo blanco, o el halo lo taparia.
assert min(IDX['marcados_clases']) > IDX['marcados']
assert [_fig.data[i].line.color for i in IDX['marcados_clases']] == COLORES_GRUPOS
assert _fig.data[IDX['marcados_sin_dato']].line.color == COLOR_MARCADO_SIN_DATO
assert all(_fig.data[i].type == 'scattergl' for i in IDX['nube_clases'] + [IDX['nube_seleccion']])

fig = go.FigureWidget(_fig)
print(f'FigureWidget con {len(fig.data)} trazas (indices 0-13 congelados, design section G)')

In [ ]:
# --- Fila 1: mapa historico con paridad 01.4 + seleccion por casilla o por clic ------
# Tres cosas que el mapa de 01.4 hace y este no hacia: se ENCUADRA sobre el circuito
# elegido (sin eso el circuito queda como un garabato diminuto en un mapa centrado en
# Manizales), dibuja transformadores e interruptores, y da hover por tramo. La cuarta es
# la seleccion: en 01.4 un vano se marca con su casilla O tocandolo en el mapa, y las dos
# vias son EL MISMO estado -- el clic alterna la casilla y deja que todo se rehaga desde
# ahi. Un registro paralelo es como la lista, el mapa y el ranking empiezan a contar
# cosas distintas.


def _seleccion_actual():
    return circuito_widget.value, ventana_widget.value, set(vano_widget.value)


def _capas_de_la_seleccion(clases_por_fid, *, campo, nombres_clase):
    """Las capas de UN mapa, con las etiquetas y el customdata que necesita el clic.

    `campo` nombra en el tooltip a que pertenece la clase -- "Criticidad original" en
    la fila 1, "Criticidad simulada" en la fila 2. Esa distincion vive ahora en el tooltip de
    cada tramo y en la leyenda, que es donde se lee mientras se mira el mapa, en vez de
    en un parrafo fijo al costado del panel.
    """
    circuito, ventana_i, marcados = _seleccion_actual()
    geo = GEO_POR_CIRCUITO.get(circuito, {'fids': [], 'lat': [], 'lon': []})
    ventana = VENTANAS[ventana_i]
    datos = DATOS_VENTANA[ventana_i]

    etiquetas = {}
    for fid in geo['fids']:
        uiti, eventos = datos.get(fid, (0.0, 0))
        clase = clases_por_fid.get(fid)
        # Sin celda en la ventana no hay clase, y eso NO es el grupo mas bajo: es la
        # ausencia del dato. Mismo criterio que el tooltip de 01.4.
        etiquetas[fid] = (
            f'<b>Vano {fid}</b><br>{ventana["etiqueta"]}: {ventana["periodo"]}'
            f'<br>{campo}: {nombres_clase[clase] if clase is not None else "sin dato"}'
            f'<br>UITI acumulado: {uiti}<br>Eventos: {eventos}'
            + ('<br>(marcado)' if fid in marcados else '')
        )
    return capas_mapa_historico(geo, clases_por_fid, marcados=marcados,
                                etiquetas_por_fid=etiquetas)


def _volcar_capa(traza, capa):
    """Las cuatro columnas van juntas SIEMPRE: si `customdata` se desfasa de lat/lon,
    Plotly desalinea el resto de la traza y el clic devuelve el vano equivocado."""
    traza.lat = capa['lat']
    traza.lon = capa['lon']
    traza.hovertext = capa['hovertext']
    traza.customdata = capa['customdata']


def _redibujar_mapa_historico(*_ignorado):
    circuito, ventana_i, _marcados = _seleccion_actual()
    capas = _capas_de_la_seleccion(clases_para(circuito, ventana_i),
                                   campo='Criticidad original', nombres_clase=NOMBRES_GRUPOS)
    seleccion = nube_seleccion(TABLA, CLASE_TABLA,
                               mask_ventana=mask_para(circuito, ventana_i),
                               marcados=_marcados)
    with fig.batch_update():
        for _clase in range(4):
            _volcar_capa(fig.data[IDX['clases'][_clase]], capas['clases'][_clase])
            _volcar_capa(fig.data[IDX['marcados_clases'][_clase]],
                         capas['marcados_por_clase'][_clase])
        _volcar_capa(fig.data[IDX['sin_dato']], capas['sin_dato'])
        _volcar_capa(fig.data[IDX['marcados']], capas['marcados'])
        _volcar_capa(fig.data[IDX['marcados_sin_dato']], capas['marcados_sin_dato'])
        # La nube: solo el resaltado se repinta, el fondo se dibuja una vez al arrancar.
        _traza_nube = fig.data[IDX['nube_seleccion']]
        _traza_nube.x = seleccion['x']
        _traza_nube.y = seleccion['y']
        _traza_nube.marker.color = [COLORES_GRUPOS[c] for c in seleccion['clase']]
        _traza_nube.hovertext = [
            f'<b>Vano {f}</b><br>Eventos: {x:,}<br>UITI acumulado: {y}'
            f'<br>Grupo: {NOMBRES_GRUPOS[c]}'
            for f, x, y, c in zip(seleccion['fid'], seleccion['x'],
                                  seleccion['y'], seleccion['clase'])
        ]


def _pintar_circuito(*_ignorado):
    """Lo que depende del CIRCUITO y no de la ventana: equipos y encuadre. Se separa del
    repintado por ventana porque mover la ventana no tiene por que recentrar el mapa --
    en 01.4 el encuadre tambien se hace una sola vez por circuito (`ULTIMO_CENTRADO`)."""
    circuito = circuito_widget.value
    tr = TRAFOS.get(circuito, {'lat': [], 'lon': []})
    sw = SWITCHES.get(circuito, {'lat': [], 'lon': []})
    vista = centro_y_zoom(GEO_POR_CIRCUITO.get(circuito, {}).get('bounds'))
    with fig.batch_update():
        for _i_tr, _i_sw in ((IDX['trafos'], IDX['switches']),
                             (IDX['pred_trafos'], IDX['pred_switches'])):
            fig.data[_i_tr].lat, fig.data[_i_tr].lon = tr['lat'], tr['lon']
            fig.data[_i_tr].hovertext = ['<b>Transformador</b>'] * len(tr['lat'])
            fig.data[_i_sw].lat, fig.data[_i_sw].lon = sw['lat'], sw['lon']
            fig.data[_i_sw].hovertext = ['<b>Interruptor / switch</b>'] * len(sw['lat'])
        if vista is not None:
            # Los dos mapas comparten encuadre a proposito: la comparacion fila 1 contra
            # fila 2 solo se sostiene si las dos miran exactamente la misma geografia.
            for _mapa in ('map', 'map2'):
                getattr(fig.layout, _mapa).center = vista['center']
                getattr(fig.layout, _mapa).zoom = vista['zoom']


_DESC = {'description_width': 'initial'}  # sin esto ipywidgets trunca los rotulos
circuito_widget = widgets.Dropdown(options=CIRCUITOS, description='Circuito',
                                   style=_DESC)
# El rotulo lleva las fechas del intervalo y no solo "V1": una ventana sin sus fechas
# obliga a ir a buscar a que periodo corresponde cada vez que se mueve el deslizador.
ventana_widget = widgets.SelectionSlider(
    options=[(f'{v["etiqueta"]}: {v["periodo"]}', v['i']) for v in VENTANAS],
    description='Ventana', continuous_update=False, style=_DESC,
    layout=widgets.Layout(width='560px'),
)
# Casillas, no SelectMultiple: es la unica forma de que un clic en el mapa alterne el
# MISMO control que el usuario ve, y de que marcar un vano no borre los ya marcados.
vano_widget = construir_selector_vanos(VANOS_POR_CIRCUITO.get(circuito_widget.value, []))


# "Marcar todos" / "Desmarcar", igual que el par de botones de 04: con circuitos de
# cientos de vanos, marcarlos de a uno no es una opcion. Los dos van por el selector y no
# por un registro propio, asi que emiten UN solo cambio de `value` y disparan un solo
# repintado -- no uno por casilla.
boton_marcar_todos = widgets.Button(description='Marcar todos', button_style='')
boton_desmarcar = widgets.Button(description='Desmarcar', button_style='')
boton_marcar_todos.on_click(lambda _b: vano_widget.marcar_todos())
boton_desmarcar.on_click(lambda _b: vano_widget.desmarcar_todos())


def _on_circuito_change(_change):
    vano_widget.poblar(VANOS_POR_CIRCUITO.get(circuito_widget.value, []))
    _pintar_circuito()
    _redibujar_mapa_historico()


def _al_hacer_clic(traza, puntos, _estado):
    """Un clic sobre un tramo alterna su vano. El fid sale de `customdata` y no del
    indice del punto: los tramos viajan concatenados con un `None` de separador, asi que
    ese indice cambia con la ventana."""
    fid = fid_de_punto(traza.customdata, getattr(puntos, 'point_inds', ()) or ())
    if fid is not None:
        vano_widget.alternar(fid)


# SOLO el mapa base. La fila 2 es la SALIDA del modelo, no un control: marcar un vano
# desde ahi mezcla "lo que yo elegi" con "lo que el modelo predijo" sobre la misma
# superficie, que es justo la confusion que separa a las dos filas (D2).
# Nota sobre el alcance del clic: plotly solo convierte un clic en evento si en ese punto
# hay hover, y en un `scattermap` de lineas el hover se calcula contra los VERTICES del
# tramo (`scattermap/hover.js`: distancia por punto, radio minimo 3 px, tope
# `layout.hoverdistance`). Hay que tocar el tramo cerca de uno de sus quiebres, no en
# cualquier parte del segmento. `hoverdistance` sube de los 20 px por defecto a 30 para
# que el blanco sea mas generoso sin llegar a marcar un vano lejano.
for _i_traza in IDX['clases'] + [IDX['sin_dato'], IDX['marcados']]:
    fig.data[_i_traza].on_click(_al_hacer_clic)
fig.layout.hoverdistance = 30

# Tier 0 del presupuesto de interactividad (design section A): elegir circuito, mover la
# ventana o marcar un vano no llama al modelo -- sin debounce ni epoch guard, que
# pertenecen al tier 1/2 (fila 2, ranking, boton "Simular"), fuera del alcance de este PR.
circuito_widget.observe(_on_circuito_change, names='value')
ventana_widget.observe(_redibujar_mapa_historico, names='value')
vano_widget.observe(_redibujar_mapa_historico, names='value')

# El fondo de la nube va una sola vez: no depende de la seleccion (01.4 ajusta el KMeans
# una vez y elegir circuito o vanos solo cambia que se resalta).
with fig.batch_update():
    for _clase in range(4):
        fig.data[IDX['nube_clases'][_clase]].x = NUBE_FONDO[_clase]['x']
        fig.data[IDX['nube_clases'][_clase]].y = NUBE_FONDO[_clase]['y']

_pintar_circuito()               # equipos y encuadre del circuito inicial
_redibujar_mapa_historico()      # primer dibujo, con la seleccion inicial

In [ ]:
# --- Importancia de variables, fila 1 col 2 (design section A, decision D7) ------------
# Es un barrido de sensibilidad min-max sobre el conjunto de muestras de los vanos
# MARCADOS del circuito y la ventana activos -- nunca SHAP (decision D5). Dos cambios
# frente a la version anterior:
#   1. Ya NO se recalcula sola al mover la ventana ni al marcar un vano, y no tiene
#      casilla "automaticas" ni boton "recalcular": corre UNA vez, dentro del mismo job
#      del boton "Simular" (celda siguiente), bajo la misma epoca. Un solo disparador
#      significa que mapa simulado e importancia siempre describen la MISMA seleccion.
#   2. Se muestra normalizada con softmax (`relevancias_015.normalizar_softmax`): la
#      participacion de cada variable, que se lee sin conocer las unidades del modelo.
#      Sin vanos marcados el grano es el circuito completo en esa ventana -- exactamente
#      las mismas filas que pinta el mapa simulado, no un panel vacio.
from chec_local_interpreter.relevancias_015 import construir_relevance_cache, fingerprint

FINGERPRINT_ACTUAL = fingerprint(
    geometrias_sha1=GEOMETRIAS_SHA1_ESPERADO,
    model_path=MODEL_PATH,
    feature_names=feature_names,
    ventana_climatica_horas=VENTANA_CLIMATICA_HORAS,
)

rankear_relevancia = construir_relevance_cache(
    model=MODELO,
    X_scaled=X,
    X_raw_model=X_raw_model,
    original_feature_df=Xdf,
    feature_names=feature_names,
    knobs=KNOBS,
    feature_scaler=feature_scaler,
    predict_fn=PREDICT_FN,
    device=DEVICE,
    context_df=context_df,
    ventanas=VENTANAS,
    fingerprint_actual=FINGERPRINT_ACTUAL,
    label_encoders=label_encoders,
    max_values_imputed=max_values_imputed,
)

RANKING_VACIO = {'vacio': True, 'filas': [], 'n_vanos': 0, 'n_filas': 0, 'mensaje': None}


def _calcular_ranking(circuito, ventana_i, marcados):
    """La parte pesada (hasta 53 pasadas del modelo, tier 2). Se llama DENTRO del job de
    "Simular" para que una sola epoca cubra el mapa y la importancia. `marcados` vacio se
    traduce a `None`, que en `rankear_relevancia` significa "todo el circuito en esta
    ventana" -- el mismo conjunto de filas del mapa simulado, y ademas la unica clave que
    aprovecha el cache en disco."""
    return rankear_relevancia(circuito, ventana_i, set(marcados) or None)


def _pintar_ranking(resultado):
    """Repaint puro, cero pasadas del modelo."""
    # Orden ascendente: en un bar horizontal Plotly dibuja la primera categoria abajo, asi
    # que la variable mas relevante (primera en `filas`, ya ordenada descendente por
    # magnitud cruda) queda arriba.
    filas = list(reversed(resultado['filas']))
    with fig.batch_update():
        fig.data[IDX['ranking']].y = [fila['label'] for fila in filas]
        fig.data[IDX['ranking']].x = [fila['relevancia'] for fila in filas]
        # La magnitud cruda no desaparece: baja al hover, que es donde se la consulta
        # cuando hace falta comparar contra otra corrida.
        fig.data[IDX['ranking']].hovertext = [
            f'<b>{fila["label"]}</b><br>Relevancia: {fila["relevancia"]:.1%}'
            f'<br>Sensibilidad min-max: {fila["magnitud_max_cambio_abs"]:.4g}'
            for fila in filas
        ]


_pintar_ranking(RANKING_VACIO)  # vacio hasta el primer "Simular"


In [ ]:
# --- Fila 2: mapa "Criticidad Simulada" + boton "Simular" (design section A, decision D2)
# El boton es el UNICO disparador del modelo y hace las dos cosas de una sola vez, bajo la
# misma epoca: 2 pasadas para el mapa simulado (`simulate_explicit_overrides`) y el barrido
# de importancia de la celda anterior. Ya no hay alternador base/simulado/delta: el mapa
# de la fila 2 muestra SIEMPRE la clase simulada, que es lo que el boton promete.
# Debounce asincronico (design section A): `asyncio.ensure_future` + cancelacion en el
# propio event loop del kernel, NUNCA `threading.Timer` -- ipykernel enruta la salida de
# los widgets con el parent header thread-local, asi que una escritura desde un hilo en
# segundo plano cae en la celda equivocada. `_EPOCA` es el guard de epoca: cualquier evento
# que invalide un job en vuelo la avanza y la escritura tardia se descarta.
from chec_local_interpreter.simulator import simulate_explicit_overrides
from chec_local_interpreter.vano_app_015 import (
    DEBOUNCE_SEGUNDOS,
    ESTADO_SIMULADO,
    aplicar_si_vigente,
    clases_por_fid_para_estado,
    construir_evento_mask_cache,
    etiqueta_capa_sin_dato,
    siguiente_epoca,
    submuestrear_si_excede,
)
from chec_local_interpreter.vano_widgets import widget_for_knob

mask_evento_para = construir_evento_mask_cache(context_df, VENTANAS)

_EPOCA = 0
_tarea_pendiente_simular = None
_ultimo_resultado_simulacion = None   # DataFrame de simulate_explicit_overrides, o None
_ultima_seleccion_simulada = None     # (circuito, ventana_i) al que corresponde ese resultado

STATUS = widgets.HTML(
    'Sin simular todavia -- elige variables (opcional) y presiona "Simular".'
)

_knobs_por_id = {k.id: k for k in KNOBS}
# Casillas y no `SelectMultiple`, por el mismo motivo que la lista de vanos: en un
# `SelectMultiple` un clic sin ctrl borra todo lo ya elegido, y aca justamente se quiere
# simular VARIAS variables a la vez. Cada casilla es independiente y `value` sigue siendo
# la tupla de knob ids, asi que `_reconstruir_controles_knob` no se entera del cambio.
knob_selector_widget = construir_selector_casillas(
    [(k.label, k.id) for k in KNOBS], titulo='', alto='150px', ancho_casilla='230px',
    layout=widgets.Layout(width='100%'),
)
controles_knob_box = widgets.VBox([])
_controles_knob_actuales = {}


def _reconstruir_controles_knob(_change=None):
    global _controles_knob_actuales
    _controles_knob_actuales = {
        knob_id: widget_for_knob(_knobs_por_id[knob_id]) for knob_id in knob_selector_widget.value
    }
    controles_knob_box.children = list(_controles_knob_actuales.values())


knob_selector_widget.observe(_reconstruir_controles_knob, names='value')

boton_simular = widgets.Button(description='Simular', button_style='primary')


def _redibujar_mapa_predicho(*_ignorado):
    """Repaint puro, CERO llamadas al modelo: usa `_ultimo_resultado_simulacion` (o
    nada, si todavia no hay simulacion para la seleccion activa)."""
    circuito, ventana_i, _marcados = _seleccion_actual()
    hay_resultado = (
        _ultimo_resultado_simulacion is not None
        and _ultima_seleccion_simulada == (circuito, ventana_i)
    )
    clases_por_fid = (
        clases_por_fid_para_estado(_ultimo_resultado_simulacion, ESTADO_SIMULADO)
        if hay_resultado else {}
    )
    # Fila 2 anti-conflation, hallazgo W1: "aun no simulado" (hay_resultado=False, TODOS los
    # vanos en sin_dato) y "simulado, sin filas de evento" (hay_resultado=True, este vano en
    # particular en sin_dato) son dos razones distintas para la misma casilla -- se distinguen
    # con color Y leyenda propios, no solo en el texto de STATUS.
    etiqueta_sin_dato = etiqueta_capa_sin_dato(hay_resultado)
    color_sin_dato = COLOR_SIN_DATO if hay_resultado else COLOR_AUN_NO_SIMULADO

    capas = _capas_de_la_seleccion(clases_por_fid, campo='Criticidad simulada',
                                   nombres_clase=NOMBRES_GRUPOS)
    with fig.batch_update():
        for _clase in range(4):
            _volcar_capa(fig.data[IDX['pred_clases'][_clase]], capas['clases'][_clase])
            fig.data[IDX['pred_clases'][_clase]].showlegend = True
        _volcar_capa(fig.data[IDX['pred_sin_dato']], capas['sin_dato'])
        fig.data[IDX['pred_sin_dato']].name = etiqueta_sin_dato
        fig.data[IDX['pred_sin_dato']].line.color = color_sin_dato
        fig.data[IDX['pred_sin_dato']].showlegend = True
        _volcar_capa(fig.data[IDX['pred_marcados']], capas['marcados'])
        fig.data[IDX['pred_marcados']].showlegend = True


def _limpiar_resultado_simulacion(_change=None):
    """Circuito o ventana cambiaron: el ultimo resultado ya NO corresponde a la
    seleccion activa -- se descarta (fila 2 vuelve a "Aun no simulado" y el panel de
    importancia se vacia) en vez de mostrar la corrida de OTRA seleccion, que violaria
    la regla anti-confusion (D2)."""
    global _ultimo_resultado_simulacion, _ultima_seleccion_simulada, _EPOCA
    _ultimo_resultado_simulacion = None
    _ultima_seleccion_simulada = None
    _EPOCA = siguiente_epoca(_EPOCA)  # invalida cualquier job en vuelo
    _redibujar_mapa_predicho()
    _pintar_ranking(RANKING_VACIO)


def _simular(epoca_job):
    """Computo pesado -- bloqueante dentro de la corutina (design section A: un job ya
    iniciado no se puede interrumpir). Mapa simulado E importancia de variables, en ese
    orden y en el mismo job. Guarda y repinta SOLO si `epoca_job` sigue vigente al
    terminar (epoch guard)."""
    global _ultimo_resultado_simulacion, _ultima_seleccion_simulada
    circuito, ventana_i, marcados = _seleccion_actual()
    mask = mask_evento_para(circuito, ventana_i)
    if not mask.any():
        aplicar_si_vigente(
            lambda: setattr(STATUS, 'value', 'Sin filas de evento para esta seleccion.'),
            epoca_job=epoca_job, epoca_actual=lambda: _EPOCA,
        )
        return

    mask_efectiva, submuestreado, n_total = submuestrear_si_excede(mask)
    overrides = expand_knob_overrides(
        {knob_id: widget.value for knob_id, widget in _controles_knob_actuales.items()}, KNOBS,
    )

    t0 = time.perf_counter()
    resultado, metadata = simulate_explicit_overrides(
        model=MODELO, X_scaled=X, X_raw_model=X_raw_model, feature_names=feature_names,
        feature_scaler=feature_scaler, predict_fn=PREDICT_FN, device=DEVICE,
        mask=mask_efectiva, vano_ids=context_df['FID_VANO'], overrides=overrides,
        label_encoders=label_encoders, max_values_imputed=max_values_imputed,
    )
    ranking = _calcular_ranking(circuito, ventana_i, marcados)
    duracion = time.perf_counter() - t0

    def _escribir():
        global _ultimo_resultado_simulacion, _ultima_seleccion_simulada
        _ultimo_resultado_simulacion = resultado
        _ultima_seleccion_simulada = (circuito, ventana_i)
        muestra = f' (muestra de {n_total:,} filas)' if submuestreado else ''
        grano = f'{len(marcados)} vanos marcados' if marcados else 'todo el circuito'
        STATUS.value = (
            f'{duracion:.2f} s | {metadata["n_vanos"]} vanos | {metadata["n_registros"]:,} filas'
            f'{muestra} | {len(overrides)} variables aplicadas | '
            f'importancia sobre {grano} ({ranking["n_filas"]:,} muestras)'
        )
        _redibujar_mapa_predicho()
        _pintar_ranking(ranking)

    aplicar_si_vigente(_escribir, epoca_job=epoca_job, epoca_actual=lambda: _EPOCA)


def _programar_simulacion(*_ignorado):
    global _EPOCA, _tarea_pendiente_simular
    if _tarea_pendiente_simular is not None and not _tarea_pendiente_simular.done():
        _tarea_pendiente_simular.cancel()
    _EPOCA = siguiente_epoca(_EPOCA)
    epoca_job = _EPOCA
    STATUS.value = 'Simulando...'

    async def _tarea():
        try:
            await asyncio.sleep(DEBOUNCE_SEGUNDOS)
        except asyncio.CancelledError:
            return
        _simular(epoca_job)

    _tarea_pendiente_simular = asyncio.ensure_future(_tarea())


boton_simular.on_click(_programar_simulacion)
circuito_widget.observe(_limpiar_resultado_simulacion, names='value')
ventana_widget.observe(_limpiar_resultado_simulacion, names='value')
vano_widget.observe(_redibujar_mapa_predicho, names='value')  # solo redibuja el halo marcado

_redibujar_mapa_predicho()  # primer dibujo: sin simulacion todavia -> "Aun no simulado"


In [ ]:
# --- El panel, ARRIBA y del ancho de la figura (paridad 01.4) -----------------------
# Una sola columna, en el orden en que se usa: circuito -> ventana -> vanos -> variables
# del simulador -> el control de cada variable elegida -> "Simular" -> estado. Cada paso
# depende del anterior, asi que apilarlos evita el zigzag de un flex-wrap donde el boton
# podia quedar antes de los deslizadores que lo alimentan.
#
# El estilo va por CSS y no por `Layout` porque ipywidgets 8 no expone `background`,
# `box-sizing` ni `gap` como traits -- solo `border`, `padding`, `margin` y el flexbox
# basico. `add_class` es la via soportada para lo demas.
ESTILO = widgets.HTML('''
<style>
  .panel-v15 {
    box-sizing: border-box;
    border-radius: 6px; background: #fdf7f6; color: #2b2b2b; font-size: 13px;
  }
  /* Cada grupo, un renglon completo: el panel es una columna, no una grilla. */
  .panel-v15 .grupo-v15 { margin: 0 0 10px 0; width: 100%; }
  .panel-v15 .titulo-v15 { font-weight: 600; margin-bottom: 2px; }
  /* La lista compacta de 01.4: letra 12px, muchas casillas por renglon y scroll propio en
     vez de estirar el panel cuando el circuito tiene cientos de vanos.
     El ancho de cada casilla NO se toca aca: viaja como estilo inline desde su `Layout`
     y le ganaria a esta hoja igual. */
  .lista-vanos, .lista-variables { font-size: 12px; }
  .lista-vanos .widget-checkbox label,
  .lista-variables .widget-checkbox label { white-space: nowrap; font-weight: 400; }
</style>''')


def _grupo(*hijos):
    """Un bloque del panel: su rotulo y sus controles juntos, como los `div` de 01.4."""
    caja = widgets.VBox(list(hijos), layout=widgets.Layout(align_items='flex-start',
                                                           width='100%'))
    caja.add_class('grupo-v15')
    return caja


def _titulo(texto):
    return widgets.HTML(f'<span class="titulo-v15">{texto}</span>')


vano_widget.caja.add_class('lista-vanos')
knob_selector_widget.caja.add_class('lista-variables')

PANEL = widgets.VBox(
    [
        _grupo(_titulo('Circuito'), circuito_widget),
        _grupo(_titulo('Ventana'), ventana_widget),
        _grupo(vano_widget,
               widgets.HBox([boton_marcar_todos, boton_desmarcar]),
               widgets.HTML('<span style="font-size:12px;color:#5b4a48;">Tambien podes '
                            'marcar y desmarcar un vano haciendo clic sobre el en el '
                            'mapa.</span>')),
        _grupo(_titulo('Variables del simulador'), knob_selector_widget),
        _grupo(controles_knob_box),
        _grupo(boton_simular),
        _grupo(STATUS),
    ],
    layout=widgets.Layout(
        width=f'{fig.layout.width}px', align_items='flex-start',
        padding='12px 14px', margin='0 0 6px 0',
        border='1px solid #e4c4c0', border_left='4px solid rgb(203,24,29)',
    ),
)
PANEL.add_class('panel-v15')

APP = widgets.VBox([ESTILO, PANEL, fig],
                   layout=widgets.Layout(width=f'{fig.layout.width}px'))


In [ ]:
display(APP)

## Como leerlo

- **El mapa de ARRIBA (fila 1) es "Criticidad Original"**, el grupo historico calculado
  por 04 sobre eventos observados -- nunca sobre una prediccion. **El de ABAJO (fila 2)
  es "Criticidad Simulada"**, la clase que predice el modelo MGCECDL con las variables
  del simulador aplicadas. No comparten leyenda ni titulo a proposito: mezclarlos
  invitaria a leer una prediccion como si fuera un hecho observado. El tooltip de cada
  tramo lo dice tramo por tramo.
- **Un vano se marca con su casilla o tocandolo en el mapa**, igual que en 04, y las
  dos vias son el mismo estado: el clic alterna la casilla. Los dos mapas aceptan clic
  porque dibujan los mismos vanos. Al elegir circuito el mapa se **encuadra** sobre el
  (centro y zoom salen de su bounding box) y aparecen sus transformadores e
  interruptores; los dos mapas comparten encuadre para que la comparacion arriba/abajo
  mire exactamente la misma geografia.
- **"Sin dato"** (gris, un color distinto de las 4 clases en ambos mapas) es un vano sin
  eventos en la ventana elegida -- no es el grupo mas bajo, es la ausencia de dato. En la
  fila 2, antes de presionar "Simular" por primera vez para la seleccion activa, el mapa
  usa un color y una leyenda DISTINTOS -- **"Aun no simulado"** (violeta) -- para no
  confundir "todavia no corri el modelo" con "corri el modelo y este vano no tiene filas
  de evento", que son dos razones distintas para la misma casilla.
- **"Simular" es el unico disparador del modelo** y hace las dos cosas en el mismo job:
  el mapa "Criticidad Simulada" (2 pasadas: base + simulado) y el panel "Importancia
  Variables". Aplica solo las variables elegidas en el panel "Variables del simulador":
  cada una aparece como un control -- deslizador para numericas, lista desplegable para
  categoricas -- y una familia climatica (precipitacion, temperatura, rafaga y viento)
  se propaga a sus 12 rezagos horarios de una sola vez.
- **"Importancia Variables"** (fila 1, columna 2) NO es SHAP: es un barrido min-max sobre
  las muestras de los vanos MARCADOS en la ventana activa -- o sobre el circuito completo
  en esa ventana si no hay ninguno marcado, que son las mismas filas del mapa simulado.
  Se muestra **normalizado con softmax**, asi que las barras suman 100% y se leen como
  participacion relativa; la magnitud cruda de sensibilidad sigue disponible en el hover.
  Ojo con la propiedad del softmax: cuando todas las magnitudes son parecidas las barras
  tienden a repartirse parejo, y eso se lee como "ninguna variable domina", no como un
  error.
- **Un vano MARCADO se dibuja con el color de SU clase**, sobre un halo blanco, igual que
  en 04 -- y en **negro** cuando no tiene celda en la ventana (sin eventos, o ausente de
  la tabla). No lleva un color plano de "seleccionado": ese color, encima de la clase,
  congelaba lo que se veia -- la ventana cambiaba la clase por debajo y el vano marcado
  seguia identico en pantalla.
- **"Grupos KMeans de vanos"** (fila 3, columna 2) es la nube de 04 traida aca: cada punto
  es una celda (vano, ventana) en el plano `(eventos, UITI acumulado)`, en el espacio
  canonico `2` (eje Y logaritmico), con los cuatro grupos de fondo y las celdas de lo
  marcado resaltadas encima. El fondo NO depende de la seleccion: 04 ajusta el KMeans una
  sola vez y elegir circuito o vanos solo cambia que se resalta. Es donde se ve directo
  por que mover la ventana cambia la clase de un vano: su punto se desplaza por el plano.
  El panel del grafo reconstruido (decision D4) sigue diferido, con su traza oculta.
- **Cambiar circuito o ventana descarta la ultima simulacion**: la fila 2 vuelve a "Aun no
  simulado" y el panel de importancia se vacia hasta la proxima vez que se presione
  "Simular" -- mostrar la corrida de OTRA seleccion violaria la misma regla
  anti-confusion de arriba.
- **Este cuaderno requiere un kernel Python vivo** -- no es exportable a HTML estatico ni
  publicable como Databricks App.